# Week 6 Exercise - Autonomous Traders (Grand Finale)

An equity trading simulation with 4 autonomous trader agents powered by MCP servers.

**Key concepts:**
1. Multi-agent system with distinct trading strategies
2. MCP server integration (accounts, market data, search, memory, notifications)
3. Custom tracing for monitoring agent activity
4. Real-time Gradio dashboard
5. Autonomous long-running agent loop

---

## Project Architecture

```
trading_floor.py          # Main loop - runs all traders every N minutes
  -> traders.py            # Trader class - connects to MCP servers, runs agent
    -> templates.py        # Prompt templates for trader/researcher instructions
    -> mcp_params.py       # MCP server configurations
    -> tracers.py          # Custom tracing -> database

accounts_server.py        # MCP Server: buy/sell/balance/holdings/change_strategy
market_server.py          # MCP Server: stock price lookup
push_server.py            # MCP Server: push notifications

accounts.py               # Account model (Pydantic) + trade execution
database.py               # SQLite persistence for accounts, logs, market data
market.py                 # Price data from Polygon.io (with fallback)

app.py                    # Gradio dashboard for real-time monitoring
reset.py                  # Initialize trader accounts with strategies
```

## Setup

Your `.env` file should include:
```
OPENAI_API_KEY=your_key_here

# Optional - market data (free tier available at polygon.io)
POLYGON_API_KEY=your_key_here
POLYGON_PLAN=free              # or: paid, realtime

# Optional - web search for researcher agent
BRAVE_API_KEY=your_key_here

# Optional - push notifications (pushover.net)
PUSHOVER_USER=your_user_here
PUSHOVER_TOKEN=your_token_here

# Optional - trading loop config
RUN_EVERY_N_MINUTES=60
RUN_EVEN_WHEN_MARKET_IS_CLOSED=False
USE_MANY_MODELS=False

# Optional - multi-model support (when USE_MANY_MODELS=True)
DEEPSEEK_API_KEY=your_key_here
GOOGLE_API_KEY=your_key_here
GROK_API_KEY=your_key_here
OPENROUTER_API_KEY=your_key_here
```

---
## Part 1: Meet the Traders

Four autonomous traders, each inspired by a titan of the industry, with distinct investment strategies.

Review their initial strategies in `reset.py`:

In [ ]:
from reset import reset_traders, waren_strategy, george_strategy, ray_strategy, cathie_strategy

traders = {
    "Warren Patience": waren_strategy,
    "George Bold": george_strategy,
    "Ray Systematic": ray_strategy,
    "Cathie Crypto": cathie_strategy,
}

for name, strategy in traders.items():
    print(f"=== {name} ===")
    print(strategy.strip())
    print()

### Initialize (or reset) trader accounts

Each trader starts with $10,000. Uncomment and run to reset.

In [ ]:
# Uncomment to reset all traders to starting positions:
# reset_traders()

---
## Part 2: MCP Server Configuration

The system uses MCP (Model Context Protocol) servers to give agents access to tools and resources.

**Trader MCP Servers** (3):
- `accounts_server.py` - buy/sell shares, check balance/holdings, change strategy
- `push_server.py` - send push notifications via Pushover
- `market_server.py` (or Polygon MCP) - stock price lookup

**Researcher MCP Servers** (3 per trader):
- `mcp-server-fetch` - fetch web pages
- `@modelcontextprotocol/server-brave-search` - web search
- `mcp-memory-libsql` - persistent memory (per-trader SQLite DB)

Review the configuration:

In [ ]:
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params
import json

print("=== Trader MCP Servers ===")
for i, params in enumerate(trader_mcp_server_params, 1):
    print(f"{i}. {params}")

print("\n=== Researcher MCP Servers (example: Warren) ===")
for i, params in enumerate(researcher_mcp_server_params("Warren"), 1):
    print(f"{i}. {params}")

---
## Part 3: The Trader Agent

Each `Trader` instance:
1. Connects to its MCP servers (accounts, market, push notifications)
2. Creates a Researcher sub-agent (with fetch, search, memory MCP servers)
3. Alternates between **trading** (finding new opportunities) and **rebalancing** (managing existing positions)
4. Executes via the OpenAI Agents SDK `Runner`

Review the key components in `traders.py`:

In [ ]:
from traders import Trader

# The Trader class orchestrates:
# 1. create_agent() - sets up Agent with researcher tool + MCP servers
# 2. run_agent()    - gets account state, picks trade/rebalance message, runs agent
# 3. run_with_mcp_servers() - manages async MCP server connections
# 4. run_with_trace() - wraps execution with custom tracing
# 5. run()          - entry point with error handling, alternates trade/rebalance

print("Trader class loaded successfully")
print(f"Max turns per run: 30")

---
## Part 4: Prompt Templates

The prompt templates in `templates.py` drive the agent behavior:

- **Researcher instructions**: search the web, use knowledge graph memory, summarize findings
- **Trader instructions**: manage portfolio according to strategy, use researcher tool, execute trades
- **Trade message**: look for new opportunities, execute trades, send push notification
- **Rebalance message**: examine existing portfolio, rebalance as needed, optionally change strategy

In [ ]:
from templates import researcher_instructions, trader_instructions, trade_message, rebalance_message

print("=== Researcher Instructions ===")
print(researcher_instructions()[:300], "...")
print()
print("=== Trader Instructions (Warren) ===")
print(trader_instructions("Warren"))

---
## Part 5: Custom Tracing

The `LogTracer` in `tracers.py` hooks into the OpenAI Agents SDK tracing system to:
- Record agent activity (trace starts/ends, span starts/ends)
- Write logs to SQLite via `database.write_log()`
- Surface trader "inner thoughts" in the dashboard

Each trace ID encodes the trader's name for routing logs to the correct trader.

In [ ]:
from tracers import LogTracer, make_trace_id

# Trace IDs encode the trader name: trace_<name>0<random>
print("Example trace IDs:")
print(f"  Warren: {make_trace_id('warren')}")
print(f"  George: {make_trace_id('george')}")
print(f"  Ray:    {make_trace_id('ray')}")
print(f"  Cathie: {make_trace_id('cathie')}")

---
## Part 6: The Trading Floor

The trading floor in `trading_floor.py` is where the magic happens:

```python
while True:
    await asyncio.gather(*[trader.run() for trader in traders])
    await asyncio.sleep(RUN_EVERY_N_MINUTES * 60)
```

All 4 traders run **in parallel** using `asyncio.gather()`, then sleep until the next cycle.

### Configuration options:
- `RUN_EVERY_N_MINUTES` - how often to run (default: 60)
- `RUN_EVEN_WHEN_MARKET_IS_CLOSED` - run outside market hours (default: False)
- `USE_MANY_MODELS` - use different LLMs per trader (default: False)

When `USE_MANY_MODELS=True`:
| Trader | Model |
|--------|-------|
| Warren | GPT 4.1 Mini |
| George | DeepSeek V3 |
| Ray    | Gemini 2.5 Flash |
| Cathie | Grok 3 Mini |

In [ ]:
from trading_floor import create_traders, RUN_EVERY_N_MINUTES, RUN_EVEN_WHEN_MARKET_IS_CLOSED, USE_MANY_MODELS

print(f"Run every: {RUN_EVERY_N_MINUTES} minutes")
print(f"Run when market closed: {RUN_EVEN_WHEN_MARKET_IS_CLOSED}")
print(f"Use many models: {USE_MANY_MODELS}")
print()

traders = create_traders()
for t in traders:
    print(f"  {t.name} {t.lastname} - model: {t.model_name}")

---
## Part 7: Running the System

The system runs as two separate processes:

### Step 1: Launch the Dashboard
Open a terminal, `cd` to the `6_mcp` directory, and run:
```bash
uv run app.py
```
This starts the Gradio web UI showing real-time trader activity, portfolios, and logs.

### Step 2: Start the Trading Floor
Open a **second** terminal, `cd` to the `6_mcp` directory, and run:
```bash
uv run trading_floor.py
```
This starts the autonomous trading loop. Watch the dashboard to see your traders in action!

### Important: Monitor API usage!
This runs continuously on a loop. Watch your API costs and stop when you've had enough.

---
## Part 8: Review the Dashboard

The Gradio dashboard (`app.py`) shows for each trader:
- Portfolio value (real-time)
- Portfolio value chart over time
- Current holdings table
- Transaction history
- Live activity logs (color-coded by type)

Log types and their meanings:
- **trace** - Agent execution lifecycle
- **agent** - Agent reasoning/decisions
- **function** - Tool calls (buy, sell, search, etc.)
- **generation** - LLM generation events
- **response** - Agent responses
- **account** - Account state changes

---
## Execution Flow Summary

```
trading_floor.py (every N minutes)
  |
  +-- asyncio.gather() runs all 4 traders in parallel
       |
       +-- Trader.run()
            |
            +-- Connect to 3 Trader MCP servers (accounts, push, market)
            +-- Connect to 3 Researcher MCP servers (fetch, search, memory)
            +-- Create Agent with Researcher as sub-tool
            +-- Get account state & strategy
            +-- Alternate: trade_message() or rebalance_message()
            +-- Runner.run(agent, message, max_turns=30)
            |    |
            |    +-- Agent uses Researcher tool for market analysis
            |    +-- Agent uses account tools to buy/sell
            |    +-- Agent uses market tools for price data
            |    +-- Agent sends push notification summary
            |
            +-- LogTracer records all activity to SQLite
            +-- Toggle do_trade flag for next cycle
```

---
## Exercise Challenges

### Challenge 1: Add a new trader
Create a 5th trader with a unique strategy (e.g., momentum trading, dividend investing, or ESG-focused). Update `reset.py` and `trading_floor.py`.

### Challenge 2: Add a new MCP server
Build a custom MCP server (e.g., sentiment analysis from social media, economic calendar, or earnings data) and integrate it with the traders.

### Challenge 3: Strategy evolution tracking
Track how traders change their own strategies over time. Add a strategy history log to the database and display it on the dashboard.

### Challenge 4: Performance competition
Add a leaderboard to the dashboard showing trader rankings by portfolio return, Sharpe ratio, or win rate.

### Challenge 5 (HARD): Inter-trader communication
Allow traders to share insights or warnings with each other through the shared memory MCP server. Have them react to each other's trades.

In [ ]:
# Scratch space for challenges
